In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from numpy import cos,pi,sqrt,log10
from astropy.table import Table
from astropy.table import unique
import os
import ctadata
from astropy.io import ascii

In [2]:
data_folder = "./LST_data/"
DL3_folder = "/pnfs/cta.cscs.ch/lst/DL3/"
run_catalog=ascii.read(data_folder + 'LST_source_catalog.ecsv')

In [3]:
Name='Mrk501'
gheff='gheff0.9'
Zdcut=30

coords_obj=SkyCoord.from_name(Name)
ra_obj=coords_obj.icrs.ra.deg
dec_obj=coords_obj.icrs.dec.deg
ra_obj,dec_obj
cdec=cos(dec_obj*pi/180.)
ra_obj,dec_obj

(253.46756952, 39.76016913)

In [5]:
#Ebins=np.logspace(log10(0.05),log10(50),16)
Ebins=np.array([0.5, 1.5, 4.0, 10.0, 25.0])
Emins=Ebins[:-1]
Emaxs=Ebins[1:]
Emeans=sqrt(Emins*Emaxs)
dE=Emaxs-Emins
Nebins=len(Emeans)

th2bins=np.linspace(0.0, 0.13, 20)
th2=(th2bins[1:]+th2bins[:-1])/2.
Nth=len(th2)


In [6]:
runlist=np.load(data_folder+'good_runs_'+Name+'_Zd_30.0.npy')
countermax=2

flist=[]

cts_s=np.zeros((Nebins,Nth))
cts_b1=np.zeros((Nebins,Nth))

tstart=np.array([])
tstop=np.array([])
texpos=np.array([])
ra_pnt=np.array([])
dec_pnt=np.array([])

runs=[]
effareas=[]
texpos=np.array([])
counter=0

for ind in range(len(runlist)):
    r=runlist[ind]
    for i in range(len(run_catalog)):
        run=run_catalog[i]['Run ID']
        if(run==r):
            date=run_catalog[i]['Date directory'].replace('-','')
            vers=ctadata.list_dir(DL3_folder+date)
            for ver in vers:
                if(ver[0]=='v'):
                    tailcuts=ctadata.list_dir(DL3_folder+date+'/'+ver)
                    for tailcut in (tailcuts):
                        nsbs=ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut)
                        for nsb in nsbs:
                            versions=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb))
                            for version in versions:
                                tags=ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std')
                                for tag in tags:
                                    src_dependences=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag))
                                    for src_dep in src_dependences:
                                        point_or_full=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep))
                                        for p_f in point_or_full:
                                            wobbles=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f))
                                            for wob in wobbles:
                                                gheffs=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob))        
                                                for gh in gheffs:
                                                    if(gheff in gh):
                                                        irfs=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh))
                                                        for irf in irfs:
                                                            files=(ctadata.list_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf))
                                                            if(run<10000):
                                                                fname='dl3_LST-1.Run0'+str(run)+'.fits'
                                                            else:
                                                                fname='dl3_LST-1.Run'+str(run)+'.fits'
                                                            if(fname in files):
                                                                flist.append(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                f=(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                counter+=1
                                                                ctadata.fetch_and_save_file_or_dir(DL3_folder+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                                hdul=fits.open(fname)
                                                                hdul=fits.open(fname)
                                                                header=hdul[1].header
                                                                dat=header['DATE-OBS']
                                                                texpos=np.concatenate((texpos,[header['LIVETIME']]))
                                                                print(ind,date,fname,sum(texpos)/3600.)
        
                                                                hdu=hdul['EFFECTIVE AREA'].data    
                                                                effareas.append(hdu['EFFAREA'][0][0]*1e4)
                
                
                                                                
                                                                ra_pnt=header['RA_PNT']
                                                                dec_pnt=header['DEC_PNT'] 
                                                                dra=ra_obj-ra_pnt
                                                                ddec=dec_obj-dec_pnt
        
                                                                ra_bkg1=ra_pnt-dra
                                                                dec_bkg1=dec_pnt-ddec
                                                                coords_bkg1=SkyCoord(ra_bkg1,dec_bkg1,unit='degree')
                    
                                                                events=hdul['EVENTS'].data
                                                                coords=SkyCoord(events['RA'],events['DEC'],unit='degree')
        
                                                                seps=coords.separation(coords_obj).deg
                                                                seps_b1=coords_bkg1.separation(coords).deg
        
                                                                energies=events['ENERGY']
                                                                i=0
                                                                for i in range(Nebins):
                                                                    m=(energies>Emins[i])*(energies<Emaxs[i])*(seps_b1>0.4) 
                                                                    h=np.histogram(seps[m]**2,bins=th2bins)
                                                                    cts_s[i]+=h[0]
                                                                    m=(energies>Emins[i])*(energies<Emaxs[i])*(seps>0.4) 
                                                                    h_b1=np.histogram(seps_b1[m]**2,bins=th2bins)
                                                                    cts_b1[i]+=h_b1[0]
        
                                                                os.remove(fname)


TokenError: Token not found in /home/renku/.config/cta-data/token. Please start agent using start-agent subcommand

In [ ]:
i=14
s_err=sqrt(cts_s[i]+cts_b1[i])
s=cts_s[i]-cts_b1[i]
plt.errorbar(th2,s,s_err,linestyle='none',marker='o',label=Name)
plt.axhline(0,color='black',linestyle='dotted')
plt.xlim(0,0.16)
plt.title(str(round(Emins[i],3))+'-'+str(round(Emaxs[i],3))+' TeV')
plt.xlabel(r'$\theta^2$, deg$^2$')
plt.ylabel('Counts')
plt.legend(loc='upper right')
#plt.savefig('th2_'+Name+'_'+str(round(Emins[i],3))+'-'+str(round(Emaxs[i],3))+'TeV.png',bbox_inches='tight')

In [ ]:
np.save('cts_'+Name+'_'+gheff,(cts_s,cts_b1))